### Setup

In [2]:
from astroquery.vizier import Vizier
import pandas as pd
import numpy as np
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.time import Time

Vizier.ROW_LIMIT = -1


In [3]:
# Replace with your own local paths and ID column names as needed

folder = '/Users/philvanlane/Documents/ax_M/'
bins = pd.read_csv(folder + 'cf_bin_dr3.csv')

In [18]:
with fits.open(folder + 'all_columns_catalog.fits', memmap=True) as hdul:
    print(hdul[1].columns.names)  # Print column names to verify

['solution_id1', 'solution_id2', 'source_id1', 'source_id2', 'random_index1', 'random_index2', 'ref_epoch1', 'ref_epoch2', 'ra1', 'ra2', 'ra_error1', 'ra_error2', 'dec1', 'dec2', 'dec_error1', 'dec_error2', 'parallax1', 'parallax2', 'parallax_error1', 'parallax_error2', 'parallax_over_error1', 'parallax_over_error2', 'pm1', 'pm2', 'pmra1', 'pmra2', 'pmra_error1', 'pmra_error2', 'pmdec1', 'pmdec2', 'pmdec_error1', 'pmdec_error2', 'ra_dec_corr1', 'ra_dec_corr2', 'ra_parallax_corr1', 'ra_parallax_corr2', 'ra_pmra_corr1', 'ra_pmra_corr2', 'ra_pmdec_corr1', 'ra_pmdec_corr2', 'dec_parallax_corr1', 'dec_parallax_corr2', 'dec_pmra_corr1', 'dec_pmra_corr2', 'dec_pmdec_corr1', 'dec_pmdec_corr2', 'parallax_pmra_corr1', 'parallax_pmra_corr2', 'parallax_pmdec_corr1', 'parallax_pmdec_corr2', 'pmra_pmdec_corr1', 'pmra_pmdec_corr2', 'astrometric_n_obs_al1', 'astrometric_n_obs_al2', 'astrometric_n_obs_ac1', 'astrometric_n_obs_ac2', 'astrometric_n_good_obs_al1', 'astrometric_n_good_obs_al2', 'astrometri

In [19]:

# Match bins against El-Badry wide binary catalog on source_id1 or source_id2

gaia_ids = set(bins['GaiaDR3_ID'].dropna().astype(np.int64))

with fits.open(folder + 'all_columns_catalog.fits', memmap=True) as hdul:
    data = hdul[1].data
    sid1 = data['source_id1'].astype(np.int64)
    sid2 = data['source_id2'].astype(np.int64)

    # Boolean mask: row matches if either source_id is in our bins list
    mask = np.isin(sid1, list(gaia_ids)) | np.isin(sid2, list(gaia_ids))

    # Extract matching rows into a DataFrame
    matched = data[mask]
    eb_matches = pd.DataFrame({col: matched[col] for col in data.columns.names})

    eb_matches = eb_matches[['source_id1', 'source_id2', 'bp_rp1','bp_rp2', 'sep_AU', 'pairdistance', 'R_chance_align']]

print(f"Found {len(eb_matches)} matching pairs for {eb_matches['source_id1'].nunique() + eb_matches['source_id2'].nunique()} unique source IDs")


Found 63 matching pairs for 126 unique source IDs


In [20]:

in1 = np.isin(eb_matches['source_id1'].values, list(gaia_ids))
in2 = np.isin(eb_matches['source_id2'].values, list(gaia_ids))
eb_matches['matched_as'] = np.select(
    [in1 & in2, in1, in2],
    ['both', 'source1', 'source2']
)


In [22]:
eb_matches.to_csv(folder + 'cf_bin_elbadry_xmatch.csv', index=False)